# MEPS data pipeline

Run the cells from top to bottom to download the approved MEPS files, build the Panel 27 person-round history, and create the study cohort and discontinuation label. Raw data and generated tables stay in Git-ignored `data/` folders.

## Find the repository

This lets the notebook run when Jupyter starts in the repository root or in `notebooks/`.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "download_meps.py").is_file()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Could not find the repository root from the notebook directory")

print(f"Repository: {ROOT}")

Repository: c:\Users\zackf\Desktop\statin-discontinuation


## 1. Download the MEPS source files

The downloader skips existing archives that pass ZIP validation.

In [2]:
subprocess.run(
    [sys.executable, str(ROOT / "src" / "download_meps.py")],
    cwd=ROOT,
    check=True,
)

CompletedProcess(args=['c:\\Users\\zackf\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'c:\\Users\\zackf\\Desktop\\statin-discontinuation\\src\\download_meps.py'], returncode=0)

## 2. Build the person-round history

Prescription and condition records are summarized before they are joined to the longitudinal data. The script writes the linked history, join audit files, and a deduplicated statin-purchase extract to `data/interim/`.

In [3]:
subprocess.run(
    [sys.executable, "-m", "src.build_person_round_history"],
    cwd=ROOT,
    check=True,
)

CompletedProcess(args=['c:\\Users\\zackf\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'src.build_person_round_history'], returncode=0)

## 3. Build the study cohort and label

This uses the linked person-round history and writes the cohort, attrition flow, label balance, quality summary, and one-row-per-person baseline dataset to `data/processed/`.

In [4]:
subprocess.run(
    [sys.executable, str(ROOT / "src" / "build_study_cohort.py")],
    cwd=ROOT,
    check=True,
)

CompletedProcess(args=['c:\\Users\\zackf\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', 'c:\\Users\\zackf\\Desktop\\statin-discontinuation\\src\\build_study_cohort.py'], returncode=0)

## Review the outputs

The person-round history contains statin cost totals by round. The baseline dataset has one row per study-cohort member, with round 1 and 2 fields reshaped to columns and the cohort label joined by person ID.

In [ ]:
import pandas as pd

history = pd.read_csv(
    ROOT / "data" / "interim" / "person_round_history.csv",
    dtype={"DUPERSID": "string"},
)
statin_purchases = pd.read_csv(
    ROOT / "data" / "interim" / "statin_purchases.csv",
    dtype={"DUPERSID": "string"},
)
baseline_dataset = pd.read_csv(
    ROOT / "data" / "processed" / "baseline_person_dataset.csv",
    dtype={"DUPERSID": "string"},
)
join_audit = pd.read_csv(ROOT / "data" / "interim" / "join_audit.csv")
unmatched_keys = pd.read_csv(ROOT / "data" / "interim" / "unmatched_linkage_keys.csv")
cohort_flow = pd.read_csv(ROOT / "data" / "processed" / "cohort_flow.csv")
label_balance = pd.read_csv(ROOT / "data" / "processed" / "label_balance.csv")

print(f"Person-round rows: {len(history):,}")
print(f"Unique statin purchases: {len(statin_purchases):,}")
print(f"Baseline people: {len(baseline_dataset):,}")
print(
    "Mean rounds 1-2 statin out-of-pocket cost: "
    f"${baseline_dataset['statin_rx_cost_baseline_total'].mean():.2f}"
)
print(
    "Purchases without a standardized drug name: "
    f"{statin_purchases['standardized_statin'].fillna('').eq('').sum():,}"
)
print(join_audit.to_string(index=False))
print(f"Unmatched source summary keys: {len(unmatched_keys):,}")
print(cohort_flow.to_string(index=False))
print(label_balance.to_string(index=False))